# SOXX/SOXL 2021 底层指数变更：Stage1 原价格面归因
按官方结构变化冻结四个时段；只读 Qlib 训练价格与成熟标签，0 拟合、0 新账户。

In [1]:
import hashlib,json,os,shlex,subprocess,tarfile,tempfile
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd().resolve();card=root/'research/stage1_index_transition_relative_return_diagnostic_20261023';plan=json.loads((card/'plan.json').read_text());source=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
sc=root/source['index_transition_card'];ss=json.loads((sc/'summary.json').read_text());assert sha(sc/'summary.json')==source['index_transition_summary_sha256'] and json.loads((sc/'result.json').read_text())['NAS_evidence']['sha256']==source['index_transition_NAS_sha256'] and ss['NAS_scoped_restore_verified']
pc=root/source['Stage1_training_card'];ps=json.loads((pc/'summary.json').read_text());assert sha(pc/'summary.json')==source['Stage1_training_summary_sha256'] and ps['recorder_id']==source['recorder_id'] and ps['nas_evidence']['sha256']==source['Stage1_training_NAS_sha256']
assert json.loads((pc/'nas_readback.json').read_text())['NAS_archive_sha256']==source['Stage1_training_NAS_sha256']
for artifact,key in [('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256')]:assert sha(pc/'qlib_recorder'/source['experiment_id']/source['recorder_id']/'artifacts'/artifact)==source[key]
assert sha(root/source['provider']/'qlib_store_manifest.json')==source['provider_manifest_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/source['provider']),region='us',kernels=1)
with R.uri_context(str(pc/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=source['recorder_id'],experiment_id=source['experiment_id']);frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
assert frame.index.equals(timing.index) and frame.index.is_unique and frame.index.is_monotonic_increasing and len(frame)==3966
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day')).normalize();assert calendar.equals(pd.DatetimeIndex(frame.index))
regimes=plan['frozen_regimes'];regime=pd.Series('outside_scope',index=frame.index)
for key,(start,end) in regimes.items():
 mask=(frame.index>=pd.Timestamp(start))&(frame.index<=pd.Timestamp(end));assert not regime.loc[mask].ne('outside_scope').any();regime.loc[mask]=key
assert (regime=='outside_scope').sum()==2
soxl=frame.soxl_forward_open_return.astype(float);soxx=frame.soxx_forward_open_return.astype(float);valid=np.isfinite(soxl)&np.isfinite(soxx)&soxl.gt(-1)&soxx.gt(-1)
residual=(soxl-3*soxx).where(valid).rename('SOXL_minus_3x_SOXX_simple_tracking_residual');relative=(np.log1p(soxl)-np.log1p(soxx)).where(valid).rename('SOXL_minus_SOXX_daily_log_relative')
parts=[relative.shift(-i) for i in range(5)];same=np.logical_and.reduce([(regime.shift(-i)==regime).to_numpy(bool) for i in range(5)]);finite=np.logical_and.reduce([np.isfinite(x.to_numpy(float)) for x in parts]);h5=pd.Series(pd.concat(parts,axis=1).sum(axis=1).where(same&finite),index=frame.index,name='SOXL_minus_SOXX_H5_log_relative')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True);training=(regime=='both_pre_change')|(regime=='both_post_change_training');held=regime=='both_post_change_reused_held';train_cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');held_cutoff=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC');h5=h5.where((training&mature.lt(train_cutoff))|(held&mature.lt(held_cutoff)))
daily=pd.DataFrame({'signal_date':frame.index,'regime':regime.values,'soxl_forward_open_return':soxl.values,'soxx_forward_open_return':soxx.values,'tracking_residual':residual.values,'daily_log_relative':relative.values,'H5_log_relative':h5.values,'H5_available_at':mature.values})
assert len(daily)==3966 and daily.signal_date.is_unique and daily.regime.value_counts().sum()==len(daily)
static=pd.concat({'feature':frame[['rv20','relative_momentum20']],'label':pd.concat([residual,relative,h5],axis=1),'meta':pd.DataFrame({'regime_code':regime.map({k:i for i,k in enumerate(list(regimes)+['outside_scope'])}),'H5_available_at':mature},index=frame.index)},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={k:tuple(v) for k,v in regimes.items()});assert sum(len(dataset.prepare(k,col_set=['feature','label','meta'])) for k in regimes)==len(daily)-2
summaries=[]
for key in regimes:
 g=daily.loc[daily.regime.eq(key)];v=g.tracking_residual.dropna();rr=g.daily_log_relative.dropna();hh=g.H5_log_relative.dropna();sr=g.soxx_forward_open_return.dropna()
 summaries.append({'regime':key,'calendar_sessions':len(g),'residual_rows':len(v),'tracking_residual_mean':float(v.mean()),'tracking_residual_median':float(v.median()),'tracking_residual_std':float(v.std(ddof=0)),'tracking_residual_abs_median':float(v.abs().median()),'tracking_residual_q05':float(v.quantile(.05)),'SOXX_simple_return_mean':float(sr.mean()),'SOXX_simple_return_std':float(sr.std(ddof=0)),'daily_log_relative_mean':float(rr.mean()),'daily_log_relative_positive_fraction':float(rr.gt(0).mean()),'H5_mature_rows':len(hh),'H5_log_relative_mean':float(hh.mean()) if len(hh) else None,'H5_log_relative_positive_fraction':float(hh.gt(0).mean()) if len(hh) else None})
annual=[{'year':int(y),'sessions':len(g),'partial_endpoint':y in (2011,2026),'tracking_residual_mean':float(g.tracking_residual.mean()),'SOXX_return_std':float(g.soxx_forward_open_return.std(ddof=0)),'H5_log_relative_mean':float(g.H5_log_relative.mean())} for y,g in daily.groupby(daily.signal_date.dt.year,sort=True)]
quarterly=[{'quarter':str(q),'sessions':len(g),'tracking_residual_mean':float(g.tracking_residual.mean()),'SOXX_return_std':float(g.soxx_forward_open_return.std(ddof=0)),'H5_log_relative_mean':float(g.H5_log_relative.mean())} for q,g in daily.groupby(daily.signal_date.dt.to_period('Q'),sort=True)]
assert len(annual)==16 and len(quarterly)==64
report={'status':'official_index_transition_relative_price_plane_descriptive_attribution_only','plan_sha256':sha(card/'plan.json'),'source_recorder_id':source['recorder_id'],'full_calendar_rows':len(daily),'regime_summary':summaries,'annual_rows':len(annual),'quarterly_rows':len(quarterly),'new_model_fits':0,'new_native_accounts':0,'bootstrap_attempts':0,'causal_effect_identified':False}
(card/'analysis_result.json').write_text(json.dumps(report,indent=2)+'\n');state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);recorder.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,source_hashes={'index_change':source['index_transition_summary_sha256'],'Stage1_frame':source['frame_artifact_sha256']});recorder.save_objects(artifact_path='analysis',daily=daily,regime_summary=summaries,annual=annual,quarterly=quarterly,analysis_result=report);recorder.log_metrics(full_calendar_rows=len(daily),pre_change_sessions=summaries[0]['calendar_sessions'],post_training_sessions=summaries[2]['calendar_sessions'],held_sessions=summaries[3]['calendar_sessions'],model_fits=0,native_accounts=0)
 with R.uri_context(str(tracking)):reread=R.get_recorder(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id)).load_object('analysis/daily')
 assert reread.equals(daily)
summary={'experiment':plan['experiment'],'decision':'descriptive_2021_common_index_transition_relative_price_plane_no_policy_change','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'analysis_result':'analysis_result.json','regime_summary':summaries,'new_model_fits':0,'new_native_accounts':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':plan['artifact_policy']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='index-transition-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert transfer.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  members=[m for m in source.getmembers() if m.isfile()];assert any(m.name.endswith('/artifacts/analysis/daily') for m in members)
  for m in members:assert (card/m.name).is_file() and source.extractfile(m).read()==(card/m.name).read_bytes()
summary['nas_evidence']=evidence;summary['nas_scoped_restore_verified']=True;(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(summary['decision'],[(x['regime'],x['calendar_sessions'],x['tracking_residual_mean'],x['tracking_residual_abs_median'],x['H5_log_relative_mean']) for x in summaries],state['recorder_id'])

2026/10/23 05:56:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[99320:MainThread](2026-10-23 05:56:47,767) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[99320:MainThread](2026-10-23 05:56:47,768) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[99320:MainThread](2026-10-23 05:56:47,769) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[99320:MainThread](2026-10-23 05:56:47,854) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_index_transition_relative_return_diagnostic_20261023.


[99320:MainThread](2026-10-23 05:56:47,856) INFO - qlib.workflow - [exp.py:258] - Experiment 332892123620548213 starts running ...


[99320:MainThread](2026-10-23 05:56:47,934) INFO - qlib.workflow - [recorder.py:345] - Recorder 5aa8d49fbc5d4b1ea08a6b0ebef7ed39 starts running under Experiment 332892123620548213 ...


[99320:MainThread](2026-10-23 05:56:49,041) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


descriptive_2021_common_index_transition_relative_price_plane_no_policy_change [('both_pre_change', 2628, -0.0001814367655023955, 0.004447996616363525, 0.003234453214318181), ('transition_unknown', 54, -0.0004276114481466788, 0.0028874874114990234, None), ('both_post_change_training', 338, -0.0002352718770856688, 0.0027604103088378906, -0.019689362594824813), ('both_post_change_reused_held', 944, -0.0005894690373186338, 0.0017505288124084473, 0.0059513057236310046)] 5aa8d49fbc5d4b1ea08a6b0ebef7ed39


## 中文结论
# SOXX/SOXL 2021 指数变更：原 Stage1 价格面归因

只读复用原 Stage1 训练价格面、成熟时间和 Qlib 日历，把 2021-06-14 至 08-27 的 **54 个交易日**保留为过渡未知。共同旧指数段 2,628 日、共同新指数但仍属于训练的 338 日、2023–2026 复用评价段 944 日全部保留；完整逐日、16 年度及 64 季度描述矩阵在 Recorder。0 拟合、0 新账户、0 区块重抽样。

简单日收益残差 `SOXL−3×SOXX` 的绝对中位数在旧域、后变更训练域、复用评价域分别约 **0.445%、0.276%、0.175%**；后两段并未显示更大的 ETF 相对跟踪偏离。同期 SOXX 简单日均收益分别为 **+0.096%、−0.052%、+0.198%**，成熟的五日 SOXL−SOXX 相对对数收益均值分别为 **+0.323%、−1.969%、+0.595%**。后变更训练段与评价段都属于共同新指数时期，但相对收益方向相反；这与市场涨跌、杠杆路径和波动影响相容，**不能把 2021 指数切换认作 Stage1 失败的因果根源**。五日标签高度重叠，这些均值不构成独立显著性或交易规则。

价格面比较没有股数、分红、Tiered 费用及账户风险反馈，不能替代原 Stage1 A_0 自有账户的 SOXX 归因，也不能通过挑选“后变更训练期”重训来救援已失败固定版本。决定：`descriptive_2021_common_index_transition_relative_price_plane_no_policy_change`。Qlib Recorder 与 Yanxuan NAS 归档身份见 summary.json，已定向读回与限定逐文件恢复。下一机制仍需在决策前预测现金机会及 SOXL 相对选择，并先过信息门和完整 0bp Tiered 自有账户。SOXX v2 目标尚未实现。
